<a href="https://colab.research.google.com/github/riyanmandanna01/GoVoyage-AI-Travel-Planner-Project-/blob/main/AI_Multi_Agent_Travel_Planner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# 🌍 GoVoyage - AI Multi-Agent Travel Planner
# ================================================================
# GOOGLE COLAB - SINGLE CELL
#
# LLM       : Gemini 3.5 Flash-Lite
# WEB       : Tavily
# FRAMEWORK : CrewAI
#
# AGENTS:
# 1. Destination Agent
# 2. Hotel Agent
# 3. Transport Agent
# 4. Restaurant Agent
# 5. Activities Agent
# 6. Itinerary Agent
#
# EXECUTION : Async CrewAI
# ================================================================


# ================================================================
# 1. INSTALL PACKAGES
# ================================================================

!pip install -q -U crewai crewai-tools tavily-python google-generativeai


# ================================================================
# 2. IMPORTS
# ================================================================

import os
import re
import json
import getpass
import asyncio
from typing import Type

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool

from pydantic import BaseModel, Field

from tavily import TavilyClient


# ================================================================
# 3. API KEYS
# ================================================================

print("=" * 70)
print("🔐 GoVoyage API Configuration")
print("=" * 70)

GEMINI_API_KEY = getpass.getpass(
    "Enter your Gemini API Key: "
).strip()

TAVILY_API_KEY = getpass.getpass(
    "Enter your Tavily API Key: "
).strip()

if not GEMINI_API_KEY:
    raise ValueError("❌ Gemini API key is required.")

if not TAVILY_API_KEY:
    raise ValueError("❌ Tavily API key is required.")


# Set environment variables
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY


# ================================================================
# 4. CONFIGURE TAVILY
# ================================================================

tavily_client = TavilyClient(
    api_key=TAVILY_API_KEY
)


# ================================================================
# 5. TAVILY SEARCH TOOL FOR CREWAI
# ================================================================

class TavilySearchInput(BaseModel):
    query: str = Field(
        ...,
        description="The live web search query."
    )


class GoVoyageTavilyTool(BaseTool):

    name: str = "GoVoyage Live Web Search"

    description: str = """
    Search the live internet using Tavily.

    Use this tool whenever current or real-world information is needed,
    including hotels, prices, trains, buses, flights, restaurants,
    tourist attractions, opening hours, travel times, reviews,
    ticket prices, availability information, and travel recommendations.

    Always perform a web search instead of relying only on memory.

    Return:
    - title
    - URL
    - published date when available
    - relevant content
    """

    args_schema: Type[BaseModel] = TavilySearchInput

    def _run(self, query: str) -> str:

        try:

            response = tavily_client.search(
                query=query,
                search_depth="advanced",
                max_results=6,
                include_answer=True,
                include_raw_content=False
            )

            output = []

            # Tavily generated answer
            answer = response.get("answer")

            if answer:
                output.append(
                    "TAVILY SUMMARY:\n" + str(answer)
                )

            # Search results
            results = response.get("results", [])

            for i, item in enumerate(results, 1):

                title = item.get(
                    "title",
                    "Untitled"
                )

                url = item.get(
                    "url",
                    ""
                )

                content = item.get(
                    "content",
                    ""
                )

                published_date = (
                    item.get("published_date")
                    or item.get("published")
                    or "Not available"
                )

                output.append(
                    f"""
SOURCE {i}
Title: {title}
URL: {url}
Published: {published_date}
Content: {content}
"""
                )

            if not output:
                return "No web results were found."

            return "\n".join(output)

        except Exception as e:

            return (
                "Tavily search failed. "
                f"Error: {str(e)}"
            )


# Create tool
web_search = GoVoyageTavilyTool()


# ================================================================
# 6. GEMINI 3.5 FLASH-LITE
# ================================================================

GEMINI_MODEL = "gemini/gemini-3.5-flash-lite"

gemini_llm = LLM(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY,
    temperature=0.2
)

print("\n✅ Gemini model configured:")
print("   ", GEMINI_MODEL)

print("✅ Tavily web search configured.")


# ================================================================
# 7. USER TRAVEL REQUIREMENTS
# ================================================================

print("\n" + "=" * 70)
print("🌍 GoVoyage Travel Planner")
print("=" * 70)

destination = input(
    "\n📍 Destination: "
).strip()

starting_location = input(
    "🚩 Starting location: "
).strip()

days = input(
    "📅 Number of days: "
).strip()

travelers = input(
    "👥 Number of travelers: "
).strip()

budget = input(
    "💰 Total budget: "
).strip()

interests = input(
    "❤️ Interests "
    "(heritage, adventure, food, nature, shopping, nightlife, etc.): "
).strip()


if not destination:
    raise ValueError("Destination cannot be empty.")

if not starting_location:
    starting_location = "Not specified"

if not days:
    days = "3"

if not travelers:
    travelers = "1"

if not budget:
    budget = "Flexible"

if not interests:
    interests = "General sightseeing"


# ================================================================
# 8. TRIP CONTEXT
# ================================================================

trip_context = f"""

GOYOVAGE TRAVEL REQUEST

Destination:
{destination}

Starting Location:
{starting_location}

Number of Days:
{days}

Number of Travelers:
{travelers}

Budget:
{budget}

Interests:
{interests}

IMPORTANT:
This is a real-world travel planning request.

Use Tavily live web search whenever current information is needed.

Do not invent:
- prices
- hotel information
- restaurant information
- transport schedules
- ticket availability
- opening hours
- URLs

If exact current information cannot be verified,
clearly say "Not verified" or "Approximate".

Preserve useful source URLs from Tavily.
"""


# ================================================================
# 9. AGENT 1 - DESTINATION AGENT
# ================================================================

destination_agent = Agent(

    role="Destination Research Specialist",

    goal=f"""
    Research {destination} using live web information and identify
    the most useful places and practical destination information
    for this specific trip.
    """,

    backstory="""
    You are an experienced travel researcher.

    You specialize in discovering:
    - major attractions
    - hidden attractions
    - heritage sites
    - natural attractions
    - cultural locations
    - local experiences
    - travel requirements
    - opening hours
    - approximate entry fees

    You always verify current information through web search.
    """,

    tools=[web_search],

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False
)


# ================================================================
# 10. AGENT 2 - HOTEL AGENT
# ================================================================

hotel_agent = Agent(

    role="Hotel and Accommodation Specialist",

    goal=f"""
    Find suitable hotels and accommodation options in {destination}
    based on the user's budget, number of travelers and trip duration.
    """,

    backstory="""
    You are a professional accommodation researcher.

    Research:
    - hotels
    - resorts
    - hostels
    - guest houses
    - family accommodation
    - budget accommodation

    Include:
    - approximate price
    - location
    - rating when available
    - facilities
    - suitability
    - source URL

    Never invent hotel prices or availability.
    """,

    tools=[web_search],

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False
)


# ================================================================
# 11. AGENT 3 - TRANSPORT AGENT
# ================================================================

transport_agent = Agent(

    role="Transportation Research Specialist",

    goal=f"""
    Research practical transportation options from
    {starting_location} to {destination}, and transportation
    within {destination}.
    """,

    backstory="""
    You are a travel transportation specialist.

    Research current information about:

    - trains
    - buses
    - flights when relevant
    - local transportation
    - taxis
    - metro
    - rental vehicles
    - estimated travel time
    - approximate fares

    Compare practical options.

    Always distinguish:
    VERIFIED information
    from
    APPROXIMATE information.

    Do not claim ticket availability unless it is actually verified.
    """,

    tools=[web_search],

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False
)


# ================================================================
# 12. AGENT 4 - RESTAURANT AGENT
# ================================================================

restaurant_agent = Agent(

    role="Food and Restaurant Specialist",

    goal=f"""
    Research restaurants and local food experiences in {destination}
    that match the user's interests and budget.
    """,

    backstory="""
    You are a food and restaurant travel specialist.

    Research:

    - restaurants
    - cafes
    - local food
    - street food
    - regional specialties
    - vegetarian options
    - popular dining areas
    - approximate price ranges
    - opening hours when available

    Provide source URLs whenever possible.

    Do not invent restaurant information.
    """,

    tools=[web_search],

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False
)


# ================================================================
# 13. AGENT 5 - ACTIVITIES AGENT
# ================================================================

activities_agent = Agent(

    role="Activities and Experiences Specialist",

    goal=f"""
    Find interesting activities and experiences in {destination}
    that match the user's interests and trip duration.
    """,

    backstory="""
    You are an expert destination activity planner.

    Research:

    - sightseeing
    - adventure activities
    - cultural experiences
    - shopping
    - nightlife
    - nature
    - museums
    - historical attractions
    - family activities
    - local experiences

    Include:

    - estimated duration
    - approximate cost
    - location
    - best time to visit
    - practical notes
    - source URL

    Verify current information with live web search.
    """,

    tools=[web_search],

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False
)


# ================================================================
# 14. TASK 1 - DESTINATION RESEARCH
# ================================================================

destination_task = Task(

    description=f"""
    {trip_context}

    Research the destination thoroughly.

    Find:

    1. Top attractions
    2. Important landmarks
    3. Cultural attractions
    4. Nature attractions
    5. Hidden/local experiences
    6. Entry fees where available
    7. Opening hours where available
    8. Approximate visit duration
    9. Important travel tips

    Use Tavily for live web research.

    Provide URLs for important sources.
    """,

    expected_output="""
    A structured destination research report containing
    attractions, costs, timings, practical information,
    and source URLs.
    """,

    agent=destination_agent
)


# ================================================================
# 15. TASK 2 - HOTEL RESEARCH
# ================================================================

hotel_task = Task(

    description=f"""
    {trip_context}

    Research accommodation options.

    Find at least 5 useful accommodation options
    across appropriate budget levels.

    For every option provide:

    - Name
    - Area/location
    - Approximate nightly price
    - Rating if available
    - Important facilities
    - Distance/proximity to attractions when available
    - Suitable traveler type
    - Source URL

    Search the live web with Tavily.

    Do not invent prices.
    """,

    expected_output="""
    A hotel comparison containing multiple accommodation
    options, approximate prices, facilities and source URLs.
    """,

    agent=hotel_agent
)


# ================================================================
# 16. TASK 3 - TRANSPORT RESEARCH
# ================================================================

transport_task = Task(

    description=f"""
    {trip_context}

    Research transportation.

    Analyze:

    1. Travel from starting location to destination
    2. Train options
    3. Bus options
    4. Flight options if useful
    5. Local transportation
    6. Estimated travel times
    7. Approximate fares
    8. Important booking considerations

    Search live web information.

    Clearly identify information that is approximate.

    Provide source URLs.
    """,

    expected_output="""
    A transportation report with route options,
    estimated times, fares and source URLs.
    """,

    agent=transport_agent
)


# ================================================================
# 17. TASK 4 - RESTAURANT RESEARCH
# ================================================================

restaurant_task = Task(

    description=f"""
    {trip_context}

    Research food and restaurant options.

    Find:

    - breakfast places
    - lunch restaurants
    - dinner restaurants
    - cafes
    - local specialties
    - street food
    - vegetarian-friendly options
    - approximate price ranges
    - locations
    - opening hours where available

    Match recommendations to the user's budget and interests.

    Use Tavily for current information.

    Include source URLs.
    """,

    expected_output="""
    A restaurant and food guide with useful recommendations,
    estimated prices and source URLs.
    """,

    agent=restaurant_agent
)


# ================================================================
# 18. TASK 5 - ACTIVITIES RESEARCH
# ================================================================

activities_task = Task(

    description=f"""
    {trip_context}

    Research activities and experiences.

    Find activities matching:

    {interests}

    Organize activities into categories such as:

    - Heritage
    - Culture
    - Nature
    - Adventure
    - Shopping
    - Food
    - Nightlife
    - Family activities

    Include:

    - approximate cost
    - duration
    - location
    - best time
    - useful travel tips
    - source URL

    Use live Tavily search.
    """,

    expected_output="""
    A categorized activity guide with prices,
    durations, practical tips and source URLs.
    """,

    agent=activities_agent
)


# ================================================================
# 19. AGENT 6 - ITINERARY AGENT
# ================================================================

itinerary_agent = Agent(

    role="Senior Travel Itinerary Planner",

    goal=f"""
    Combine all research into one practical,
    realistic and detailed GoVoyage travel plan
    for {destination}.
    """,

    backstory="""
    You are the senior travel planner responsible
    for producing the final customer-ready itinerary.

    You combine research from destination,
    accommodation, transportation, food and activity specialists.

    You prioritize:

    - logical routes
    - realistic travel times
    - user's interests
    - budget
    - traveler count
    - practical scheduling
    - verified information
    - source transparency

    You may perform additional Tavily searches whenever
    the specialist reports contain missing or uncertain
    current information.
    """,

    tools=[web_search],

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False
)


# ================================================================
# 20. FINAL ITINERARY TASK
# ================================================================

itinerary_task = Task(

    description=f"""
    {trip_context}

    Create the final GoVoyage travel plan.

    You have access to the following specialist research:

    DESTINATION RESEARCH:
    {{destination_research}}

    HOTEL RESEARCH:
    {{hotel_research}}

    TRANSPORT RESEARCH:
    {{transport_research}}

    RESTAURANT RESEARCH:
    {{restaurant_research}}

    ACTIVITIES RESEARCH:
    {{activities_research}}


    ============================================================
    FINAL OUTPUT FORMAT
    ============================================================

    # 🌍 GoVoyage Travel Plan

    ## 1. Trip Overview

    Include:

    - Destination
    - Starting location
    - Number of days
    - Number of travelers
    - Budget
    - Travel interests


    ## 2. 🏨 Recommended Hotels

    Provide a table:

    | Hotel | Area | Approx. Price | Rating | Highlights | Source |
    |-------|------|---------------|--------|------------|--------|

    Clearly mark prices as approximate when they are not
    directly verified.


    ## 3. 🚆 Transportation

    Include:

    ### Getting There

    - Train
    - Bus
    - Flight if relevant

    For each:

    - approximate fare
    - travel time
    - practical notes
    - source URL

    ### Local Transportation

    Explain:

    - taxi
    - metro
    - bus
    - rental options
    - walking


    ## 4. 🏛️ Places to Visit

    Organize attractions by priority and category.

    Include:

    - estimated time
    - entry fee where available
    - opening information
    - source


    ## 5. 🍴 Food & Restaurants

    Include:

    - breakfast
    - lunch
    - dinner
    - cafes
    - local specialties
    - street food

    Include approximate prices and sources.


    ## 6. 🎯 Activities

    Match activities with:

    {interests}

    Include:

    - estimated duration
    - approximate cost
    - location
    - best time
    - source


    ## 7. 📅 Day-by-Day Itinerary

    Create a detailed itinerary.

    DAY 1
    Morning:
    Afternoon:
    Evening:
    Food:
    Transportation:
    Estimated daily cost:

    DAY 2
    Morning:
    Afternoon:
    Evening:
    Food:
    Transportation:
    Estimated daily cost:

    Continue for every day.


    ## 8. 💰 Estimated Budget

    Create an estimated budget:

    | Category | Estimated Cost |
    |----------|----------------|
    | Transport | |
    | Hotels | |
    | Food | |
    | Activities | |
    | Local Transport | |
    | Miscellaneous | |
    | TOTAL | |


    Separate:

    - per-person estimate
    - total group estimate


    ## 9. 💡 Travel Tips

    Include useful advice about:

    - weather
    - clothing
    - local transport
    - safety
    - booking
    - timings
    - local customs


    ## 10. 🔗 SOURCES

    Provide a clean list of all important source URLs.

    Example:

    - Hotel: https://...
    - Transport: https://...
    - Attraction: https://...
    - Restaurant: https://...


    ============================================================
    IMPORTANT ACCURACY RULES
    ============================================================

    1. Do not fabricate URLs.

    2. Do not fabricate prices.

    3. Do not claim ticket availability unless verified.

    4. Distinguish approximate prices from verified prices.

    5. Prefer official websites where available.

    6. Use Tavily for missing current information.

    7. Keep source URLs in the final answer.

    8. Make the itinerary geographically practical.

    9. Avoid scheduling too many attractions in one day.

    10. Respect the user's budget and interests.
    """,

    expected_output="""
    A complete customer-ready GoVoyage travel plan containing:

    - trip overview
    - hotels
    - transportation
    - attractions
    - restaurants
    - activities
    - day-by-day itinerary
    - budget
    - travel tips
    - source URLs
    """,

    agent=itinerary_agent,

    context=[
        destination_task,
        hotel_task,
        transport_task,
        restaurant_task,
        activities_task
    ]
)


# ================================================================
# 21. CREATE CREW
# ================================================================

crew = Crew(

    agents=[
        destination_agent,
        hotel_agent,
        transport_agent,
        restaurant_agent,
        activities_agent,
        itinerary_agent
    ],

    tasks=[
        destination_task,
        hotel_task,
        transport_task,
        restaurant_task,
        activities_task,
        itinerary_task
    ],

    process=Process.sequential,

    verbose=True
)


# ================================================================
# 22. ASYNC EXECUTION
# ================================================================

print("\n")
print("=" * 70)
print("🚀 Starting GoVoyage Multi-Agent Travel Research")
print("=" * 70)

print("\nAgents:")
print("  1. 🏛️ Destination Agent")
print("  2. 🏨 Hotel Agent")
print("  3. 🚆 Transport Agent")
print("  4. 🍴 Restaurant Agent")
print("  5. 🎯 Activities Agent")
print("  6. 🗺️ Itinerary Agent")

print("\n🌐 Live Tavily web research enabled")
print("🧠 Gemini 3.5 Flash-Lite enabled")
print("⚡ Async CrewAI execution enabled")

print("\n" + "=" * 70)


# IMPORTANT:
# Google Colab already runs an event loop.
# Therefore use await directly.
#
# DO NOT use asyncio.run(...)
# DO NOT use crew.kickoff(...)

result = await crew.kickoff_async()


# ================================================================
# 23. FINAL RESULT
# ================================================================

print("\n\n")
print("=" * 80)
print("🌍 GOYOVAGE FINAL TRAVEL PLAN")
print("=" * 80)

print(result)


# ================================================================
# 24. EXTRACT SOURCE URLS
# ================================================================

try:

    final_text = str(result)

    urls = re.findall(
        r'https?://[^\s\]\)\}>"\'\,]+',
        final_text
    )

    # Clean URLs
    cleaned_urls = []

    for url in urls:

        url = url.rstrip(
            ".,;:!?)]}>\"'"
        )

        if url not in cleaned_urls:
            cleaned_urls.append(url)


    print("\n\n")
    print("=" * 80)
    print("🔗 GOYOVAGE SOURCE URLS")
    print("=" * 80)

    if cleaned_urls:

        for i, url in enumerate(cleaned_urls, 1):
            print(f"{i}. {url}")

    else:

        print(
            "No URLs were detected in the final response."
        )

except Exception as e:

    print(
        "\n⚠️ Could not extract source URLs:",
        str(e)
    )


# ================================================================
# 25. COMPLETION MESSAGE
# ================================================================

print("\n")
print("=" * 80)
print("✅ GoVoyage planning completed!")
print("=" * 80)

print(
    "\nPowered by:"
    "\n• CrewAI Multi-Agent System"
    "\n• Gemini 3.5 Flash-Lite"
    "\n• Tavily Live Web Search"
)

🔐 GoVoyage API Configuration
Enter your Gemini API Key: ··········
Enter your Tavily API Key: ··········

✅ Gemini model configured:
    gemini/gemini-3.5-flash-lite
✅ Tavily web search configured.

🌍 GoVoyage Travel Planner

📍 Destination: Delhi
🚩 Starting location: kanyakumari
📅 Number of days: 15
👥 Number of travelers: 20
💰 Total budget: 1000
❤️ Interests (heritage, adventure, food, nature, shopping, nightlife, etc.): food


🚀 Starting GoVoyage Multi-Agent Travel Research

Agents:
  1. 🏛️ Destination Agent
  2. 🏨 Hotel Agent
  3. 🚆 Transport Agent
  4. 🍴 Restaurant Agent
  5. 🎯 Activities Agent
  6. 🗺️ Itinerary Agent

🌐 Live Tavily web research enabled
🧠 Gemini 3.5 Flash-Lite enabled
⚡ Async CrewAI execution enabled



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5f9f82ca-8e79-4389-9c22-d4cc4c3162dd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  GOYOVAGE TRAVEL REQUEST                                                                                        │
│                                                                                                                 │
│  Destination:                                                                                                   │
│  Delhi                                                                                                          │
│                                                                                                                 │
│  Starting Location:                                                                                             │
│  kanyakumari                                                                                                    │
│                                                                                                                 │
│  Number of Days:                                                                                                │
│  15                                                                                                             │
│                                                                                                                 │
│  Number of Travelers:                                                                                           │
│  20                                                                                                             │
│                                                                                                                 │
│  Budget:                                                                                                        │
│  1000                                                                                                           │
│                                                                                                                 │
│  Interests:                                                                                                     │
│  food                                                                                                           │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  This is a real-world travel planning request.                                                                  │
│                                                                                                                 │
│  Use Tavily live web search whenever current information is needed.                                             │
│                                                                                                                 │
│  Do not invent:                                                                                                 │
│  - prices                                                                                                       │
│  - hotel information                                                                                            │
│  - restaurant information                              

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Destination Research Specialist                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  GOYOVAGE TRAVEL REQUEST                                                                                        │
│                                                                                                                 │
│  Destination:                                                                                                   │
│  Delhi                                                                                                          │
│                                                                                                                 │
│  Starting Location:                                                                                             │
│  kanyakumari                                                                                                    │
│                                                                                                                 │
│  Number of Days:                                                                                                │
│  15                                                                                                             │
│                                                                                                                 │
│  Number of Travelers:                                                                                           │
│  20                                                                                                             │
│                                                                                                                 │
│  Budget:                                                                                                        │
│  1000                                                                                                           │
│                                                                                                                 │
│  Interests:                                                                                                     │
│  food                                                                                                           │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  This is a real-world travel planning request.                                                                  │
│                                                                                                                 │
│  Use Tavily live web search whenever current information is needed.                                             │
│                                                                                                                 │
│  Do not invent:                                                                                                 │
│  - prices                                                                                                       │
│  - hotel information                                   

╭──────────────────────────────────────── 🔧 Tool Execution Started (#13) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: go_voyage_live_web_search                                                                                │
│  Args: {'query': 'Delhi major tourist attractions entry fee timing'}                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Travel Itinerary Planner                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  GOYOVAGE TRAVEL REQUEST                                                                                        │
│                                                                                                                 │
│  Destination:                                                                                                   │
│  Delhi                                                                                                          │
│                                                                                                                 │
│  Starting Location:                                                                                             │
│  kanyakumari                                                                                                    │
│                                                                                                                 │
│  Number of Days:                                                                                                │
│  15                                                                                                             │
│                                                                                                                 │
│  Number of Travelers:                                                                                           │
│  20                                                                                                             │
│                                                                                                                 │
│  Budget:                                                                                                        │
│  1000                                                                                                           │
│                                                                                                                 │
│  Interests:                                                                                                     │
│  food                                                                                                           │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  This is a real-world travel planning request.                                                                  │
│                                                                                                                 │
│  Use Tavily live web search whenever current information is needed.                                             │
│                                                                                                                 │
│  Do not invent:                                                                                                 │
│  - prices                                                                                                       │
│  - hotel information                                   

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Travel Itinerary Planner                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 🌍 GoVoyage Travel Plan                                                                                      │
│                                                                                                                 │
│  ## 1. Trip Overview                                                                                            │
│                                                                                                                 │
│  - **Destination:** Delhi, India                                                                                │
│  - **Starting Location:** Kanyakumari, India                                                                    │
│  - **Number of Days:** 15 Days                                                                                  │
│  - **Number of Travelers:** 20 People (Group/Family Tour)                                                       │
│  - **Budget:** $1,000 USD total (Approx. ₹83,000–₹85,000 INR total; ultra-budget constraint requiring strict    │
│  management of public transit, street food, and free/low-cost public attractions)                               │
│  - **Travel Interests:** Food & Culinary Exploration                                                            │
│                                                                                                                 │
│                                                                                                                 │
│  ## 2. 🏨 Recommended Hotels                                                                                    │
│                                                                                                                 │
│  | Hotel | Area | Approx. Price | Rating | Highlights | Source |                                                │
│  |-------|------|---------------|--------|------------|--------|                                                │
│  | Zostel Delhi | Paharganj, Central Delhi | ₹800 – ₹2,200 INR (~$10–$26) per bed/room | 8.1 / 10 | Free        │
│  Wi-Fi, air conditioning, in-house cafe, central transit access | [Zostel                                       │
│  Delhi](https://www.zostel.com/destination/delhi/stay/delhi-dlhh524) |                                          │
│  | Joey's Hostel Delhi | Laxmi Nagar, East Delhi | ₹600 – ₹1,800 INR (~$7–$22) per bed | Highly rated | Shared  │
│  kitchen, rooftop terrace, close to metro station |                                                             │
│  [Booking.com](https://www.booking.com/hotel/in/joey-39-s-hostel.en.html) |                                     │
│  | The Hosteller Delhi | Friends Colony East, South East Delhi | ₹700 – ₹2,500 INR (~$8–$30) per bed/room |     │
│  Very good | Modern backpacker amenities, social areas, near Humayun's Tomb | [The Hosteller                    │
│  Delhi](https://www.thehosteller.com/hostels/the-hosteller-delhi) |                                             │
│  | Hotel Ajanta | Arakashan Road, Paharganj | ₹1,500 – ₹3,500 INR (~$20–$42) per private room | Highly rated    │
│  budget hotel | Private rooms, multi-cuisine restaurant, rooftop terrace | [Hotel                               │
│  Ajanta](http://www.ajantahoteldelhi.com) |                                                                     │
│                                                          

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  GOYOVAGE TRAVEL REQUEST                                                                                        │
│                                                                                                                 │
│  Destination:                                                                                                   │
│  Delhi                                                                                                          │
│                                                                                                                 │
│  Starting Location:                                                                                             │
│  kanyakumari                                                                                                    │
│                                                                                                                 │
│  Number of Days:                                                                                                │
│  15                                                                                                             │
│                                                                                                                 │
│  Number of Travelers:                                                                                           │
│  20                                                                                                             │
│                                                                                                                 │
│  Budget:                                                                                                        │
│  1000                                                                                                           │
│                                                                                                                 │
│  Interests:                                                                                                     │
│  food                                                                                                           │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  This is a real-world travel planning request.                                                                  │
│                                                                                                                 │
│  Use Tavily live web search whenever current information is needed.                                             │
│                                                                                                                 │
│  Do not invent:                                                                                                 │
│  - prices                                                                                                       │
│  - hotel information                                                                                            │
│  - restaurant information                              




🌍 GOYOVAGE FINAL TRAVEL PLAN
# 🌍 GoVoyage Travel Plan

## 1. Trip Overview

- **Destination:** Delhi, India
- **Starting Location:** Kanyakumari, India
- **Number of Days:** 15 Days
- **Number of Travelers:** 20 People (Group/Family Tour)
- **Budget:** $1,000 USD total (Approx. ₹83,000–₹85,000 INR total; ultra-budget constraint requiring strict management of public transit, street food, and free/low-cost public attractions)
- **Travel Interests:** Food & Culinary Exploration


## 2. 🏨 Recommended Hotels

| Hotel | Area | Approx. Price | Rating | Highlights | Source |
|-------|------|---------------|--------|------------|--------|
| Zostel Delhi | Paharganj, Central Delhi | ₹800 – ₹2,200 INR (~$10–$26) per bed/room | 8.1 / 10 | Free Wi-Fi, air conditioning, in-house cafe, central transit access | [Zostel Delhi](https://www.zostel.com/destination/delhi/stay/delhi-dlhh524) |
| Joey's Hostel Delhi | Laxmi Nagar, East Delhi | ₹600 – ₹1,800 INR (~$7–$22) per bed | Highly rated | Shared ki